In [9]:
import pandas as pd
import statsmodels.api as sm

# 전진선택법 (5분 컷)
def forward_selection(x, y):
    x = sm.add_constant(x) #절편추가
    selected = ['const']
    remaining = [c for c in x.columns if c != 'const'] # x 컬럼이 남아있는지 확인
    
    while remaining:
        best_aic, best_var = float('inf'), None #초기화
        for var in remaining: #컬럼을 하나씩 꺼냄
            try:
                aic = sm.OLS(y, x[selected + [var]]).fit(disp=0).aic #bic
                if aic < best_aic: #bic
                    best_aic, best_var = aic, var #갱신
            except:
                continue
        
        current_aic = sm.OLS(y, x[selected]).fit(disp=0).aic #bic
        if best_aic < current_aic:
            selected.append(best_var)
            remaining.remove(best_var)
        else:
            break
    
    return selected, sm.OLS(y, x[selected]).fit(disp=0) #선택한 컬럼명과 모델 출력

# 후진소거법 (5분 컷)
def backward_elimination(x, y):
    x = sm.add_constant(x)
    selected = list(x.columns)
    
    while len(selected) > 1:
        best_aic, worst_var = float('inf'), None
        for var in selected:
            if var == 'const':
                continue
            try:
                temp = [v for v in selected if v != var]
                aic = sm.OLS(y, x[temp]).fit(disp=0).aic
                if aic < best_aic:
                    best_aic, worst_var = aic, var
            except:
                continue
        
        current_aic = sm.OLS(y, x[selected]).fit(disp=0).aic
        if best_aic < current_aic:
            selected.remove(worst_var)
        else:
            break
    
    return selected, sm.OLS(y, x[selected]).fit(disp=0)

# 단계적선택법 (5분 컷)
def stepwise_selection(x, y):
    x = sm.add_constant(x)
    selected = ['const']
    all_vars = [c for c in x.columns if c != 'const']
    
    for _ in range(len(all_vars)):
        # Forward
        remaining = [v for v in all_vars if v not in selected]
        if remaining:
            best_aic, best_var = float('inf'), None
            for var in remaining:
                try:
                    aic = sm.OLS(y, x[selected + [var]]).fit(disp=0).aic
                    if aic < best_aic:
                        best_aic, best_var = aic, var
                except:
                    continue
            
            current_aic = sm.OLS(y, x[selected]).fit(disp=0).aic
            if best_aic < current_aic:
                selected.append(best_var)
        
        # Backward
        if len(selected) > 1:
            best_aic, worst_var = float('inf'), None
            for var in selected:
                if var == 'const':
                    continue
                try:
                    temp = [v for v in selected if v != var]
                    aic = sm.OLS(y, x[temp]).fit(disp=0).aic
                    if aic < best_aic:
                        best_aic, worst_var = aic, var
                except:
                    continue
            
            current_aic = sm.OLS(y, x[selected]).fit(disp=0).aic
            if best_aic < current_aic:
                selected.remove(worst_var)
    
    return selected, sm.OLS(y, x[selected]).fit(disp=0)

# 사용법
# vars, model = forward_selection(X, y)
# vars, model = backward_elimination(X, y) 
# vars, model = stepwise_selection(X, y)
# print(f"선택변수: {vars}, AIC: {model.aic:.4f}")

In [10]:
dt= pd.read_csv('https://raw.githubusercontent.com/algoboni/pythoncodebook1-1/main/practice8_BHP2.csv')
dt.info() #여기서 집값을 종속변수로, 나머지를  설명변수로 선택

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7495 entries, 0 to 7494
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     7495 non-null   object 
 1   availability  7495 non-null   int64  
 2   size          7495 non-null   int64  
 3   total_sqft    7495 non-null   float64
 4   bath          7495 non-null   int64  
 5   balcony       7495 non-null   int64  
 6   price         7495 non-null   float64
dtypes: float64(2), int64(4), object(1)
memory usage: 410.0+ KB


In [12]:
dt = pd.get_dummies(dt, columns=dt.select_dtypes('O').columns, drop_first=True)

x = dt.drop(columns=['price'])
y = dt['price']

#전진선택법
feat, model =  forward_selection(x, y)

print(f"선택변수: {feat}")
print(model.summary())

선택변수: ['const', 'total_sqft', 'bath', 'area_type_Plot']
                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.521
Model:                            OLS   Adj. R-squared:                  0.520
Method:                 Least Squares   F-statistic:                     2711.
Date:                Thu, 31 Jul 2025   Prob (F-statistic):               0.00
Time:                        16:29:42   Log-Likelihood:                -42788.
No. Observations:                7495   AIC:                         8.558e+04
Df Residuals:                    7491   BIC:                         8.561e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------

In [13]:
#후진소거  
feat, model =  backward_elimination(x, y)

print(f"선택변수: {feat}")
print(model.summary())

선택변수: ['const', 'total_sqft', 'bath', 'area_type_Plot']
                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.521
Model:                            OLS   Adj. R-squared:                  0.520
Method:                 Least Squares   F-statistic:                     2711.
Date:                Thu, 31 Jul 2025   Prob (F-statistic):               0.00
Time:                        16:30:27   Log-Likelihood:                -42788.
No. Observations:                7495   AIC:                         8.558e+04
Df Residuals:                    7491   BIC:                         8.561e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------

In [14]:
#단계선택
feat, model =  stepwise_selection(x, y)

print(f"선택변수: {feat}")
print(model.summary())

선택변수: ['const', 'total_sqft', 'bath', 'area_type_Plot']
                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.521
Model:                            OLS   Adj. R-squared:                  0.520
Method:                 Least Squares   F-statistic:                     2711.
Date:                Thu, 31 Jul 2025   Prob (F-statistic):               0.00
Time:                        16:31:34   Log-Likelihood:                -42788.
No. Observations:                7495   AIC:                         8.558e+04
Df Residuals:                    7491   BIC:                         8.561e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------

In [15]:
print(model.params)

const            -62.421561
total_sqft         0.053314
bath              30.974968
area_type_Plot    83.233767
dtype: float64
